# Embedding distillation scaling laws: quality 3-hour A100 budget

Focused study of metric vs synthetic dataset size. Default scope: AG News, BERT base, soft labels, CLS attention labels, and real-example embedding initialization. This version moves the run closer to the paper configuration while keeping a three-hour A100 budget. Run the pilot first, then inspect the suggested outer-step budget before starting the grid.

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import json
import math
import os
import sys
import time
import traceback
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    ROOT = cwd
elif (cwd.parent / "src").exists():
    ROOT = cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {cwd}")

sys.path.insert(0, str(ROOT / "src"))

from dataset_attrs import DATASET_ATTRS
from distillation import run_embedding_distillation

sns.set_theme(style="whitegrid")
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "cpu")


## Scope

This notebook intentionally studies one axis: synthetic dataset size. It now uses the higher-quality embedding distillation setting from the paper-inspired path: trainable soft labels, CLS attention labels, real-example embedding initialization, paper-default train batch size, and a longer synthetic sequence length. Keeping architecture, dataset and label mode fixed makes the scaling curve interpretable while spending the A100 budget on optimization rather than on a broad hyperparameter sweep.

In [ ]:
TASK = "ag_news"
MODEL_NAME = "bert-base-uncased"
LABEL_TYPE = "soft"
ATTENTION_LABEL_TYPE = "cls"
INIT_STRATEGY = "real"

DPC_GRID = [1, 2, 3, 5, 10, 20]
SEEDS = [42, 43, 44]

# The paper default is 512. 256 is a practical middle ground for the
# heavier CLS-attention-label setting under a single 3-hour A100 run.
SEQ_LENGTH = 256
INNER_STEPS = 2
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 256
N_EVAL_MODEL = 10

BUDGET_HOURS = 3.0
RESERVE_MINUTES = 20
PILOT_DPC = max(DPC_GRID)
PILOT_OUTER_STEPS = 30
OUTER_STEPS_CAP = 7200
SKIP_COMPLETED = True

SETTING_SLUG = f"{LABEL_TYPE}_attn-{ATTENTION_LABEL_TYPE}_init-{INIT_STRATEGY}_seq-{SEQ_LENGTH}_inner-{INNER_STEPS}"
SCALING_ROOT = ROOT / "results" / "embedding_scaling_laws_3h" / TASK / MODEL_NAME.replace("/", "__") / SETTING_SLUG
SCALING_ROOT.mkdir(parents=True, exist_ok=True)

N_RUNS = len(DPC_GRID) * len(SEEDS)
print("runs:", N_RUNS)
print("dpc:", DPC_GRID)
print("seeds:", SEEDS)
print("setting:", SETTING_SLUG)
print("results:", SCALING_ROOT)


## Pilot

The pilot runs the most expensive DPC with thirty outer updates and a tiny evaluation slice. Its estimate is deliberately conservative because model loading, real-example initialization, and preprocessing are included.

In [ ]:
pilot_dir = SCALING_ROOT / "_pilot"
t0 = time.perf_counter()
pilot_data, pilot_metrics = run_embedding_distillation(
    task_name=TASK,
    model_name=MODEL_NAME,
    dpc=PILOT_DPC,
    seq_length=SEQ_LENGTH,
    inner_steps=INNER_STEPS,
    train_epochs=1,
    batch_size_per_label=PILOT_DPC,
    label_type=LABEL_TYPE,
    attention_label_type=ATTENTION_LABEL_TYPE,
    init_strategy=INIT_STRATEGY,
    save_dir=pilot_dir,
    data_root=ROOT / "data",
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    max_train_batches_per_epoch=PILOT_OUTER_STEPS,
    max_eval_batches=1,
    n_eval_model=1,
    seed=42,
    bf16=torch.cuda.is_available(),
)
pilot_elapsed_sec = time.perf_counter() - t0

seconds_per_outer_step = pilot_elapsed_sec / PILOT_OUTER_STEPS
usable_seconds = BUDGET_HOURS * 3600 - RESERVE_MINUTES * 60 - pilot_elapsed_sec
suggested_outer_steps = max(1, min(OUTER_STEPS_CAP, math.floor(usable_seconds / N_RUNS / seconds_per_outer_step)))

print("pilot elapsed sec:", round(pilot_elapsed_sec, 1))
print("conservative sec / outer step:", round(seconds_per_outer_step, 2))
print("suggested OUTER_STEPS:", suggested_outer_steps)
print("pilot metrics are timing-only:", pilot_metrics)

del pilot_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Confirm Budget

The suggested value is capped at 7200 outer updates. Lower it if you need more reserve. Keep the same value for every DPC: changing optimization budget together with dataset size would confound the scaling-law curve.

In [ ]:
OUTER_STEPS = suggested_outer_steps
RUN_BUDGET_HOURS = max(0.0, (BUDGET_HOURS * 3600 - RESERVE_MINUTES * 60 - pilot_elapsed_sec) / 3600)
ESTIMATED_GRID_HOURS = seconds_per_outer_step * OUTER_STEPS * N_RUNS / 3600

print("outer steps per run:", OUTER_STEPS)
print("estimated grid hours:", round(ESTIMATED_GRID_HOURS, 2))
print("grid budget hours:", RUN_BUDGET_HOURS)


In [ ]:
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_dir(dpc: int, seed: int) -> Path:
    return SCALING_ROOT / f"dpc{dpc}_seed{seed}"


def summary_path(dpc: int, seed: int) -> Path:
    return run_dir(dpc, seed) / "summary.json"


def save_rows(rows: list[dict]):
    df = pd.DataFrame(rows)
    df.to_csv(SCALING_ROOT / "scaling_laws.csv", index=False)
    return df


## Run Scaling Grid

Each completed run is checkpointed as `summary.json`. Re-running this cell resumes from completed combinations. The loop stops before starting a new run once the time budget is exhausted.

In [ ]:
grid_started = time.perf_counter()
deadline = grid_started + RUN_BUDGET_HOURS * 3600
rows = []
errors = []
stop_requested = False

for dpc in DPC_GRID:
    for seed in SEEDS:
        path = summary_path(dpc, seed)
        if SKIP_COMPLETED and path.exists():
            rows.append(json.loads(path.read_text()))
            print("skip existing:", path)
            continue

        completed_durations = [row["elapsed_sec"] for row in rows if "elapsed_sec" in row]
        estimated_next_sec = max(completed_durations[-3:], default=seconds_per_outer_step * OUTER_STEPS)
        if time.perf_counter() + estimated_next_sec >= deadline:
            print("insufficient budget for the next run")
            stop_requested = True
            break

        rd = run_dir(dpc, seed)
        rd.mkdir(parents=True, exist_ok=True)
        print(f"\n=== dpc={dpc} seed={seed} outer_steps={OUTER_STEPS} ===")

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            synthetic_data, metrics = run_embedding_distillation(
                task_name=TASK,
                model_name=MODEL_NAME,
                dpc=dpc,
                seq_length=SEQ_LENGTH,
                inner_steps=INNER_STEPS,
                train_epochs=1,
                batch_size_per_label=dpc,
                label_type=LABEL_TYPE,
                attention_label_type=ATTENTION_LABEL_TYPE,
                init_strategy=INIT_STRATEGY,
                save_dir=rd,
                data_root=ROOT / "data",
                train_batch_size=TRAIN_BATCH_SIZE,
                eval_batch_size=EVAL_BATCH_SIZE,
                max_train_batches_per_epoch=OUTER_STEPS,
                max_eval_batches=None,
                n_eval_model=N_EVAL_MODEL,
                seed=seed,
                bf16=torch.cuda.is_available(),
            )
            elapsed_sec = time.perf_counter() - t0
            num_synthetic_examples = synthetic_data.num_examples
            train_size = 120000 if TASK == "ag_news" else None
            row = {
                "task_name": TASK,
                "model_name": MODEL_NAME,
                "label_type": LABEL_TYPE,
                "attention_label_type": ATTENTION_LABEL_TYPE,
                "init_strategy": INIT_STRATEGY,
                "dpc": dpc,
                "seed": seed,
                "num_synthetic_examples": num_synthetic_examples,
                "compression_ratio": train_size / num_synthetic_examples if train_size else None,
                "seq_length": SEQ_LENGTH,
                "inner_steps": INNER_STEPS,
                "train_batch_size": TRAIN_BATCH_SIZE,
                "eval_batch_size": EVAL_BATCH_SIZE,
                "n_eval_model": N_EVAL_MODEL,
                "outer_steps": OUTER_STEPS,
                "outer_steps_cap": OUTER_STEPS_CAP,
                "elapsed_sec": elapsed_sec,
                "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None,
                **metrics,
            }
            path.write_text(json.dumps(row, indent=2))
            rows.append(row)
            save_rows(rows)
            print({"accuracy": round(row["accuracy"], 4), "macro_f1": round(row["macro_f1"], 4), "minutes": round(elapsed_sec / 60, 1)})
        except Exception as exc:
            error = {
                "dpc": dpc,
                "seed": seed,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
            (rd / "error.json").write_text(json.dumps(error, indent=2))
            errors.append(error)
            print("FAILED", error["error_type"], error["error"])
        finally:
            cleanup_cuda()

    if stop_requested:
        break

df = save_rows(rows)
print("completed:", len(df), "/", N_RUNS)
print("errors:", len(errors))
df


## Collect Saved Results

In [ ]:
rows = [json.loads(path.read_text()) for path in SCALING_ROOT.glob("dpc*_seed*/summary.json")]
df = pd.DataFrame(rows)
if not df.empty:
    df = df.sort_values(["dpc", "seed"])
    df.to_csv(SCALING_ROOT / "scaling_laws.csv", index=False)
df


## Aggregate and Plot

In [ ]:
if df.empty:
    raise RuntimeError("No summaries found. Run the grid first.")

agg = (
    df.groupby(["dpc", "num_synthetic_examples"], as_index=False)
      .agg(
          accuracy_mean=("accuracy", "mean"),
          accuracy_std=("accuracy", "std"),
          macro_f1_mean=("macro_f1", "mean"),
          macro_f1_std=("macro_f1", "std"),
          elapsed_sec_mean=("elapsed_sec", "mean"),
          peak_gpu_gb_mean=("peak_gpu_gb", "mean"),
          runs=("seed", "count"),
      )
)
agg.to_csv(SCALING_ROOT / "scaling_laws_aggregated.csv", index=False)
agg


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].errorbar(agg["num_synthetic_examples"], agg["accuracy_mean"], yerr=agg["accuracy_std"], marker="o", capsize=3)
axes[0].set_xscale("log")
axes[0].set_xlabel("Synthetic examples")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy scaling")

axes[1].errorbar(agg["num_synthetic_examples"], agg["macro_f1_mean"], yerr=agg["macro_f1_std"], marker="o", capsize=3)
axes[1].set_xscale("log")
axes[1].set_xlabel("Synthetic examples")
axes[1].set_ylabel("Macro-F1")
axes[1].set_title("Macro-F1 scaling")

axes[2].plot(agg["num_synthetic_examples"], agg["elapsed_sec_mean"] / 60, marker="o")
axes[2].set_xscale("log")
axes[2].set_xlabel("Synthetic examples")
axes[2].set_ylabel("Minutes per run")
axes[2].set_title("Runtime scaling")

fig.suptitle(f"Embedding distillation scaling laws: {TASK}, {MODEL_NAME}, {SETTING_SLUG}")
fig.tight_layout()
plot_path = SCALING_ROOT / "scaling_laws.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
print("saved", plot_path)


## Simple Error Power-Law Fit

This is a descriptive log-log fit: `error ≈ a * N^b`. With only six points it should be reported as an exploratory estimate, not as a definitive asymptotic law.

In [ ]:
fit_df = agg.dropna(subset=["accuracy_mean"]).copy()
fit_df["error"] = 1.0 - fit_df["accuracy_mean"]
fit_df = fit_df[fit_df["error"] > 0]

slope, intercept = np.polyfit(np.log(fit_df["num_synthetic_examples"]), np.log(fit_df["error"]), deg=1)
a = float(np.exp(intercept))
fit_summary = {
    "formula": "error ~= a * num_synthetic_examples ** b",
    "a": a,
    "b": float(slope),
    "num_points": len(fit_df),
}
(SCALING_ROOT / "power_law_fit.json").write_text(json.dumps(fit_summary, indent=2))
fit_summary
